Name: Hussain Badri
Id: 2025AF05063
MLOPS Assignment 2

In [1]:
import time
import pandas as pd
import ollama

MODEL = "llama3.2:3b"

questions = [
    {"id": 1, "text": "What is the difference between supervised and unsupervised learning?"},
    {"id": 2, "text": "Explain photosynthesis in simple terms."},
    {"id": 3, "text": "What causes inflation in an economy?"},
    {"id": 4, "text": "Describe the water cycle and its main stages."},
    {"id": 5, "text": "What is the difference between DNA and RNA?"},
]

latency_results = []

for q in questions:
    prompt = f"Answer the following student question clearly.\n\nQuestion: {q['text']}"
    
    start = time.time()
    response = ollama.generate(model=MODEL, prompt=prompt, stream=False)
    end = time.time()
    
    latency_ms = (end - start) * 1000
    
    latency_results.append({
        "question_id": q["id"],
        "prompt_version": "v1",
        "model": MODEL,
        "start_time": start,
        "end_time": end,
        "latency_ms": round(latency_ms, 2),
        "total_duration_ns": response.get("total_duration"),
        "status": "success"
    })

df_latency = pd.DataFrame(latency_results)
print(df_latency)

print("\n--- Latency Statistics (ms) ---")
print(df_latency["latency_ms"].describe())

C:\Users\Lenovo\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


   question_id prompt_version        model    start_time      end_time  \
0            1             v1  llama3.2:3b  1.793717e+09  1.793717e+09   
1            2             v1  llama3.2:3b  1.793717e+09  1.793717e+09   
2            3             v1  llama3.2:3b  1.793717e+09  1.793717e+09   
3            4             v1  llama3.2:3b  1.793717e+09  1.793717e+09   
4            5             v1  llama3.2:3b  1.793717e+09  1.793717e+09   

   latency_ms  total_duration_ns   status  
0    15486.67        15376519000  success  
1     7905.40         7888242500  success  
2    10254.56        10210808000  success  
3    15847.14        15807777400  success  
4     7887.84         7875800300  success  

--- Latency Statistics (ms) ---
count        5.000000
mean     11476.322000
std       3946.781298
min       7887.840000
25%       7905.400000
50%      10254.560000
75%      15486.670000
max      15847.140000
Name: latency_ms, dtype: float64


In [2]:
df_latency.to_csv("latency_tokens_cost.csv", index=False)

The slowest request was question 3 (inflation), with a latency of approximately X ms. This is likely because the generated response contained more tokens (higher eval_count) than other questions, requiring more generation time. Local inference speed depends on CPU/GPU capability, model size, and output length; the first request may also include model loading overhead (cold start).

In [3]:
COMPUTE_COST_PER_HOUR = 0.10  # Document this assumption in your report
COMPUTE_COST_PER_MS = COMPUTE_COST_PER_HOUR / 3_600_000

cost_results = []

for q in questions:
    prompt = f"Answer the following student question clearly.\n\nQuestion: {q['text']}"
    
    start = time.time()
    response = ollama.generate(model=MODEL, prompt=prompt, stream=False)
    latency_ms = (time.time() - start) * 1000
    
    input_tokens = response.get("prompt_eval_count", 0)
    output_tokens = response.get("eval_count", 0)
    total_tokens = input_tokens + output_tokens
    
    cost = COMPUTE_COST_PER_MS * latency_ms
    
    cost_results.append({
        "question_id": q["id"],
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
        "cost_usd": round(cost, 8)
    })

df_cost = pd.DataFrame(cost_results)
print(df_cost)

avg_cost = df_cost["cost_usd"].mean()
projected_monthly = avg_cost * 10_000
print(f"\nAverage cost per request: ${avg_cost:.8f}")
print(f"Projected monthly cost (10,000 requests): ${projected_monthly:.4f}")

   question_id  input_tokens  output_tokens  total_tokens  cost_usd
0            1            46            354           400  0.000351
1            2            41            265           306  0.000246
2            3            41            349           390  0.000325
3            4            43            434           477  0.000399
4            5            43            276           319  0.000256

Average cost per request: $0.00031537
Projected monthly cost (10,000 requests): $3.1537


In [4]:
df_combined = df_latency.merge(df_cost, on="question_id")
df_combined.to_csv("latency_tokens_cost.csv", index=False)

Recommendation: Since cost is driven by compute time (latency), reduce latency by (a) using a smaller model like qwen2.5:0.5b for simple factual questions, (b) capping output length with Ollama's num_predict parameter to avoid unnecessarily long responses, or (c) shortening the system prompt to reduce prompt_eval_count. Prompt caching for repeated system instructions can also reduce input token processing time.

In [5]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("edaussist_llmops_assignment")

2026/11/03 09:49:59 INFO mlflow.tracking.fluent: Experiment with name 'edaussist_llmops_assignment' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1793717399303, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1793717399303, lifecycle_stage='active', name='edaussist_llmops_assignment', tags={}, trace_location=None, workspace='default'>

In [6]:
# Version 1 - Basic
prompt_v1 = mlflow.genai.register_prompt(
    name="edaussist-academic-qa",
    template="Answer the following student question clearly.\n\nQuestion: {{question}}",
    commit_message="v1 - Basic prompt"
)
print(f"v1 registered as version {prompt_v1.version}")

# Version 2 - Structured
prompt_v2 = mlflow.genai.register_prompt(
    name="edaussist-academic-qa",
    template="You are an academic assistant. Answer the following student question in a clear, concise, and accurate manner. Use simple language suitable for undergraduate students.\n\nQuestion: {{question}}",
    commit_message="v2 - Structured prompt"
)
print(f"v2 registered as version {prompt_v2.version}")

# Version 3 - JSON Output
v3_template = """You are an academic assistant. Answer the student question and return ONLY valid JSON. Do not include markdown, explanations outside JSON, or code fences.

Return this exact structure:
{"answer": "string", "confidence": "high | medium | low", "key_points": ["string"]}

Question: {{question}}"""

prompt_v3 = mlflow.genai.register_prompt(
    name="edaussist-academic-qa",
    template=v3_template,
    commit_message="v3 - JSON output prompt"
)
print(f"v3 registered as version {prompt_v3.version}")

2026/11/03 09:54:05 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: edaussist-academic-qa, version 1


v1 registered as version 1


2026/11/03 09:54:07 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: edaussist-academic-qa, version 2
2026/11/03 09:54:08 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: edaussist-academic-qa, version 3


v2 registered as version 2
v3 registered as version 3


In [7]:
import json

def is_valid_json(text):
    try:
        data = json.loads(text)
        return 1 if all(k in data for k in ["answer", "confidence", "key_points"]) else 0
    except (json.JSONDecodeError, TypeError):
        return 0

eval_rows = []

for version_label, prompt_obj in [("v1", prompt_v1), ("v2", prompt_v2), ("v3", prompt_v3)]:
    for q in questions:
        formatted_prompt = prompt_obj.format(question=q["text"])
        
        response = ollama.generate(model=MODEL, prompt=formatted_prompt, stream=False)
        output = response["response"]
        
        row = {
            "prompt_version": version_label,
            "question_id": q["id"],
            "output": output,
            "input_tokens": response.get("prompt_eval_count", 0),
            "output_tokens": response.get("eval_count", 0),
        }
        
        if version_label == "v3":
            row["valid_json"] = is_valid_json(output)
        
        eval_rows.append(row)

df_eval = pd.DataFrame(eval_rows)
df_eval.to_csv("prompt_evaluation_results.csv", index=False)
print(df_eval.groupby("prompt_version").agg({"input_tokens": "mean", "output_tokens": "mean"}))

                input_tokens  output_tokens
prompt_version                             
v1                      42.8          362.4
v2                      64.8          319.8
v3                      93.8           93.4


In [8]:
with mlflow.start_run(run_name="prompt_evaluation"):
    v3_rows = df_eval[df_eval["prompt_version"] == "v3"]
    mean_valid = v3_rows["valid_json"].mean()
    mlflow.log_metric("valid_json_v3_mean", mean_valid)
    
    for version_label in ["v1", "v2", "v3"]:
        subset = df_eval[df_eval["prompt_version"] == version_label]
        mlflow.log_metric(f"avg_input_tokens_{version_label}", subset["input_tokens"].mean())
        mlflow.log_metric(f"avg_output_tokens_{version_label}", subset["output_tokens"].mean())

🏃 View run prompt_evaluation at: http://localhost:5000/#/experiments/1/runs/c33d89c925c34200aac43f1d8a8a630c
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [9]:
best_version = 3  # Adjust based on your evaluation results

mlflow.genai.set_prompt_alias(
    name="edaussist-academic-qa",
    alias="production",
    version=best_version
)
print(f"Production alias set to version {best_version}")

Production alias set to version 3
